<a href="https://colab.research.google.com/github/bobaboiz0127/bobaboiz0127.github.io/blob/main/Week%205/ITOM6219_HW5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 5 — A/B Testing

In this homework, you will apply the A/B testing workflow covered in Week 5. You will retrieve data from the Google Analytics API, perform hypothesis testing using Student's and Welch's t-tests, and interpret the results.

In [2]:
!pip3 install google.analytics.data
!pip3 install pingouin

In [3]:
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest,
    Filter,
    FilterExpression,
)
from pathlib import Path
import os
import pandas as pd
import scipy.stats as st
import numpy as np
from pingouin import ttest

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_credentials_path():
    """Return the path to the Google Analytics service-account JSON file."""
    if userdata is not None:
        try:
            secret_path = userdata.get("GOOGLE_CREDENTIALS_PATH")
            if secret_path:
                return secret_path
        except Exception:
            pass

    candidate_paths = [
        "/mnt/data/bigquery-419120-14d56b3f70eb (1).json",
        "./bigquery-419120-14d56b3f70eb (1).json",
        "./bigquery-419120-14d56b3f70eb.json",
    ]

    for candidate in candidate_paths:
        if os.path.exists(candidate):
            return candidate

    json_matches = list(Path("/mnt/data").glob("*.json")) + list(Path(".").glob("*.json"))
    if json_matches:
        return str(json_matches[0])

    raise FileNotFoundError(
        "Could not find the Google Analytics JSON credentials file. "
        "Upload the JSON file and place it in the working directory."
    )


# Project 1: Campaign Comparison

Compare the session numbers for two campaigns: one with medium "announcement" and the other with medium "canvas". You will retrieve data from Google Analytics, prepare the data, and perform statistical tests to compare the difference.

Please make sure to upload the JSON credentials file to the working folder.
Add a secret key:
- **Name:** `GOOGLE_CREDENTIALS_PATH`
- **Value:** name of the JSON file

## Problem 1.1

Construct the function `sample_run_report`:
- Keep dimensions: `source`, `medium`, `campaignName`, `date`
- Keep metric: `sessions`
- Date range: `start_date="2026-02-01"`, `end_date="today"`
- `property_id="424145747"` (This is Jane's app)

Expected output:
- Code

> 💡 **Hint:** See class notebook **Section 2.3.1** for an example of constructing a Google Analytics report function with dimensions, metrics, and date ranges. See lecture notes **Section 2.3.3** for the full API code pattern.

In [4]:
def sample_run_report(property_id="424145747"):
    """Runs a Google Analytics 4 sessions report for the two campaign mediums."""
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = get_credentials_path()
    client = BetaAnalyticsDataClient()

    request = RunReportRequest(
        property=f"properties/{property_id}",
        dimensions=[
            Dimension(name="source"),
            Dimension(name="medium"),
            Dimension(name="campaignName"),
            Dimension(name="date"),
        ],
        metrics=[Metric(name="sessions")],
        date_ranges=[DateRange(start_date="2026-02-01", end_date="today")],
        dimension_filter=FilterExpression(
            filter=Filter(
                field_name="medium",
                in_list_filter=Filter.InListFilter(values=["canvas", "announcement"]),
            )
        ),
    )

    response = client.run_report(request)
    return response


## Problem 1.2
Convert the resulting data into a dataframe.

Expected output:
- Code
- Print the output (screenshot)

> 💡 **Hint:** See class notebook **Section 2.3.3** for the `response_to_df` helper function that converts the API response into a pandas DataFrame.

In [5]:
def response_to_df(response):
    columns = []
    rows = []

    for col in response.dimension_headers:
        columns.append(col.name)
    for col in response.metric_headers:
        columns.append(col.name)

    for row_data in response.rows:
        row = []
        for val in row_data.dimension_values:
            row.append(val.value)
        for val in row_data.metric_values:
            row.append(val.value)
        rows.append(row)

    return pd.DataFrame(rows, columns=columns)


response = sample_run_report(property_id="424145747")
df = response_to_df(response)
print(df)


          source        medium campaignName      date sessions
0         social  announcement      evening  20260312        7
1         social  announcement    afternoon  20260312        6
2         social  announcement    afternoon  20260311        4
3         social  announcement    afternoon  20260317        3
4   announcement        canvas    (not set)  20260305        2
5   announcement        canvas    (not set)  20260317        2
6         canvas  announcement           v1  20260319        2
7         canvas  announcement           v1  20260409        2
8         canvas  announcement           v2  20260409        2
9         social  announcement    afternoon  20260305        2
10        social  announcement    afternoon  20260310        2
11        social  announcement      evening  20260310        2
12        social  announcement      evening  20260311        2
13        social  announcement      evening  20260314        2
14        social  announcement      evening  20260317  

## Problem 1.3
Prepare the data by extracting the session numbers for each campaign medium into separate variables.

Note: Please use medium being canvas and medium being announcement to identify two campaigns.


Expected output:
- Code

> 💡 **Hint:** See class notebook **Section 2.4.2** for how to filter a DataFrame by condition and extract a column for each group (control vs. treatment).

In [6]:
canvas = df[df["medium"] == "canvas"]["sessions"].astype(int)
announcement = df[df["medium"] == "announcement"]["sessions"].astype(int)

print("Canvas observations:", len(canvas))
print("Announcement observations:", len(announcement))
print("Canvas mean sessions:", round(canvas.mean(), 3))
print("Announcement mean sessions:", round(announcement.mean(), 3))


Canvas observations: 8
Announcement observations: 27
Canvas mean sessions: 1.25
Announcement mean sessions: 2.0


## Problem 1.4
Run Student's t-test and Welch's t-test to compare the two campaign mediums.

Expected output:
- Code
- Conclusion

> 💡 **Hint:** See class notebook **Section 2.4.3** for running both Student's t-test (`correction=False`) and Welch's t-test (`correction=True`) using `pingouin`. See lecture notes **Section 2.5.1** and **Section 2.5.2** for interpreting the results.

In [7]:
student_results = ttest(canvas, announcement, correction=False)
welch_results = ttest(canvas, announcement, correction=True)

print("Student's t-test")
print(student_results)
print("\nWelch's t-test")
print(welch_results)

student_p = student_results["p_val"].iloc[0]
welch_p = welch_results["p_val"].iloc[0]

equal_var_check = np.isclose(canvas.var(ddof=1), announcement.var(ddof=1), rtol=0.25)
same_sample_size = len(canvas) == len(announcement)

print("\nInterpretation")
print(f"- Canvas mean sessions: {canvas.mean():.3f}")
print(f"- Announcement mean sessions: {announcement.mean():.3f}")
print(f"- Student's t-test p-value: {student_p:.4f}")
print(f"- Welch's t-test p-value: {welch_p:.4f}")

if student_p < 0.05:
    print("- Student's t-test: the two channel means are statistically different at the 5% level.")
else:
    print("- Student's t-test: the two channel means are not statistically different at the 5% level.")

if welch_p < 0.05:
    print("- Welch's t-test: the two channel means are statistically different at the 5% level.")
else:
    print("- Welch's t-test: the two channel means are not statistically different at the 5% level.")

if equal_var_check and same_sample_size:
    print("- The two tests should be very similar here because the groups have similar variance and the same sample size.")
else:
    print("- Welch's t-test is more appropriate here because it does not assume equal variances.")


Student's t-test
               T  dof alternative    p_val           CI95   cohen_d     power  \
T_test -1.387563   33   two-sided  0.17457  [-1.85, 0.35]  0.558547  0.270528   

        BF10  
T_test  0.74  

Welch's t-test
               T        dof alternative     p_val            CI95   cohen_d  \
T_test -2.267457  32.790133   two-sided  0.030081  [-1.42, -0.08]  0.558547   

           power   BF10  
T_test  0.270528  2.273  

Interpretation
- Canvas mean sessions: 1.250
- Announcement mean sessions: 2.000
- Student's t-test p-value: 0.1746
- Welch's t-test p-value: 0.0301
- Student's t-test: the two channel means are not statistically different at the 5% level.
- Welch's t-test: the two channel means are statistically different at the 5% level.
- Welch's t-test is more appropriate here because it does not assume equal variances.


## Problem 1.5
Compare the tests and draw conclusions.

> 💡 **Hint:** See lecture notes **Section 2.5.3** for the distinction between statistical and practical significance, and **Section 2.6** for how to frame a launch decision.

### Your Output — Problem 1.5

```
+------------------------------------------------------------+
| Compare the Student's t-test and Welch's t-test results.   |
| Do they agree? Which is more appropriate here and why?      |
|                                                            |
|                                                            |
+------------------------------------------------------------+
```

---
> 📤 **Submit to Canvas — Problem 1.5:** Submit your comparison and conclusions.
---

## Problem 1.6
From the [Google Analytics API schema](https://developers.google.com/analytics/devguides/reporting/data/v1/api-schema), choose a metric that is different from `sessions`. Explain why your chosen metric is of significance and run statistical tests to tell whether these two channels are different regarding your chosen metric.

Expected output:
- Code
- Explanation of your chosen metric and why it matters
- Statistical test results and interpretation

> 💡 **Hint:** See lecture notes **Section 2.2.1** for how to select a meaningful metric (measurable, attributable, sensitive, timely). See class notebook **Section 2.3.1** for how to modify the `metrics` parameter in the API request.

In [15]:
def sample_run_report_metric(metric_name="engagedSessions", property_id="424145747"):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = get_credentials_path()
    client = BetaAnalyticsDataClient()

    request = RunReportRequest(
        property=f"properties/{property_id}",
        dimensions=[
            Dimension(name="source"),
            Dimension(name="medium"),
            Dimension(name="campaignName"),
            Dimension(name="date"),
        ],
        metrics=[Metric(name=metric_name)],
        date_ranges=[DateRange(start_date="2026-02-01", end_date="today")],
        dimension_filter=FilterExpression(
            filter=Filter(
                field_name="medium",
                in_list_filter=Filter.InListFilter(values=["canvas", "announcement"]),
            )
        ),
    )

    return client.run_report(request)

metric_name = "engagedSessions"
response_metric = sample_run_report_metric(metric_name=metric_name, property_id="424145747")
df_metric = response_to_df(response_metric)
df_metric[metric_name] = df_metric[metric_name].astype(int)

canvas_metric = df_metric[df_metric["medium"] == "canvas"][metric_name]
announcement_metric = df_metric[df_metric["medium"] == "announcement"][metric_name]

student_metric = ttest(canvas_metric, announcement_metric, correction=False)
welch_metric = ttest(canvas_metric, announcement_metric, correction=True)

print("Student's t-test")
print(student_metric)
print("\nWelch's t-test")
print(welch_metric)

Student's t-test
               T  dof alternative     p_val          CI95   cohen_d     power  \
T_test -1.330688   33   two-sided  0.192414  [-1.8, 0.38]  0.535653  0.252689   

       BF10  
T_test  0.7  

Welch's t-test
               T        dof alternative     p_val            CI95   cohen_d  \
T_test -2.170185  32.745273   two-sided  0.037349  [-1.38, -0.04]  0.535653   

           power  BF10  
T_test  0.252689  1.96  


### Your Output — Problem 1.6

```
+------------------------------------------------------------+
- I chose **engagedSessions**.
- This metric matters because it measures session quality, not just traffic volume.
- A channel can bring many sessions, but if users leave quickly or do not interact, that traffic is less valuable.
- Use the p-values from the Student's and Welch's t-tests above to decide whether the two channels are statistically different on engaged sessions.
- For reporting, Welch's t-test is the better default because campaign groups may have different variances.                       |
|                                                            |
+------------------------------------------------------------+
```

---
> 📤 **Submit to Canvas — Problem 1.6:** Submit your code, chosen metric justification, and test interpretation.
---

# Project 2: Simulated A/B Tests

In this project, you will perform statistical tests on simulated data to see whether two samples have the same mean. This will help you understand how sample size, variance, and test assumptions affect your conclusions.

## Problem 2.1
The two samples below have the same level of variance (standard deviation = 5) and sample size but different means (25 and 30).

Based on the data, run both Student's t-test and Welch's t-test.

Expected output:
- Are these two samples' means statistically different?
- Do you get different results from Student's t-test and Welch's t-test?

> 💡 **Hint:** See class notebook **Section 2.4.3** for running Student's t-test (`correction=False`) and Welch's t-test (`correction=True`). See lecture notes **Section 2.5.1** and **Section 2.5.2** for interpreting the two tests. When both groups have equal variance, which test should give similar results?

In [9]:
import numpy as np
from scipy import stats
np.random.seed(67)

# Simulated data for the control group (the mean is 25, the standard deviation is 5, and the sample size is 1000)
control_group = np.random.normal(loc=25, scale=5, size=1000) #

# Simulated data for the treatment group (the mean is 30, the standard deviation is 5, and the sample size is 1000)
treatment_group = np.random.normal(loc=30, scale=5, size=1000)

In [10]:
# Student's T-test
from pingouin import ttest

ttest(treatment_group,control_group,correction=False)

,T,dof,alternative,p_val,CI95,cohen_d,power,BF10
T_test,23.082351,1998,two-sided,1.072302e-104,"[4.75, 5.63]",1.032274,1.0,6.582e+100


In [11]:
# Welch's T-test
from pingouin import ttest

ttest(treatment_group, control_group, correction=True)


,T,dof,alternative,p_val,CI95,cohen_d,power,BF10
T_test,23.082351,1997.980237,two-sided,1.072577e-104,"[4.75, 5.63]",1.032274,1.0,6.582e+100


### Your Output — Problem 2.1

```
+------------------------------------------------------------+
| Are these two samples' means statistically different?      |
| Do you get different results from the two tests?           |
|                                                            |
|                                                            |
+------------------------------------------------------------+
```

---
> 📤 **Submit to Canvas — Problem 2.1:** Submit your test results and answers.
---

## Problem 2.2
The two samples below have different sample sizes, means, and variances.

Based on the data, run both Student's t-test and Welch's t-test.

Expected output:
- Are these two samples' means statistically different?
- Do you get different results from Student's t-test and Welch's t-test? Which test is more reliable? Why?

> 💡 **Hint:** See lecture notes **Section 2.5.2** — Welch's t-test relaxes the equal-variance assumption. When sample sizes and variances differ substantially, Student's t-test may produce misleading results. Think about which assumption is violated here.

In [12]:
np.random.seed(8690978)
control_group = np.random.normal(loc=25, scale=3, size=60)
treatment_group = np.random.normal(loc=30, scale=20, size=100)

In [13]:
# Student's T-test
from pingouin import ttest
ttest(treatment_group,control_group,correction=False)

,T,dof,alternative,p_val,CI95,cohen_d,power,BF10
T_test,1.626181,158,two-sided,0.105904,"[-0.98, 10.07]",0.265554,0.365723,0.589


In [14]:
# Welch's T-test
from pingouin import ttest
ttest(treatment_group, control_group, correction=True)

,T,dof,alternative,p_val,CI95,cohen_d,power,BF10
T_test,2.081504,105.10609,two-sided,0.039817,"[0.22, 8.88]",0.265554,0.365723,1.268


### Your Output — Problem 2.2

```
+------------------------------------------------------------+
| Do you get different results from Student's t-test and     |
| Welch's t-test? Which test is more reliable? Why?          |
|                                                            |
|                                                            |
|                                                            |
+------------------------------------------------------------+
```

---
> 📤 **Submit to Canvas — Problem 2.2:** Submit your test results and explanation of which test is more reliable.
---